# Algoritmos Genéticos

Estrutura básica do algoritmo evolutivo:

```
1. t = 0
2. Inicializar população P0
3. Enquanto critério de parada == falso
   3.1 Avaliar população (Pt)
   3.2 P’ = Selecionar pais (Pt)
   3.3 F  = Aplicar recombinação e mutação (P’)
   3.4 Avaliar população (F)
   3.5 Pt+1 = Selecionar sobreviventes(Pt + F)
   3.6 t = t + 1
```


## Modelagem do cromossomo

In [ ]:
# Classe que representa um cromossomo binário para o algoritmo genético
# Utiliza a biblioteca dataclass para simplificar a criação de objetos
# O vetor é inicializado com valores binários aleatórios (0 ou 1)
# A função de fitness é uma função passada como parâmetro e usada para avaliar a qualidade do cromossomo
from dataclasses import dataclass
from dataclasses import field
from typing import List
from typing import Callable
from random import randint


@dataclass
class Cromossomo:
    fitness: Callable[[int], float]
    tamanho: int = 0
    vetor: List[int] = field(default_factory=list)

    def __post_init__(self):
        """Método chamado após o construtor."""
        self.vetor = [randint(0, 1) for _ in range(self.tamanho)]

    def __getitem__(self, key):
        """Sobrecarga do operador de indexação."""
        return self.vetor[key]

    def __setitem__(self, key, value):
        """Sobrecarga do operador de indexação."""
        self.vetor[key] = value

    def __len__(self):
        return self.tamanho

    def get_fitness(self):
        return self.fitness(self.vetor)

    # TODO: Implementar a cache do fitness


In [ ]:
# Exemplo de instanciamento de um cromossomo com 10 genes binários
# A função lambda passada calcula o fitness como a soma dos valores binários (maior número de 1s => melhor)
v = Cromossomo(lambda x: sum(x), 10)
print(v.vetor)

[0, 1, 0, 0, 1, 0, 1, 0, 1, 0]


## Algoritmos de seleção dos pais

In [ ]:
# Função de seleção aleatória com reposição
# A cada iteração, dois cromossomos são escolhidos aleatoriamente para formar um casal
# Essa abordagem não considera o valor de fitness e serve como base de comparação com métodos mais sofisticados
from typing import Tuple
from random import sample

# Função de seleção de pais recebe a população (lista com todos cromossomos)
# e ela retorna uma lista com os casais selecionados

def selecao_aleatoria_com_reposicao(P: List[Cromossomo]) -> List[Tuple[Cromossomo, Cromossomo]]:
    pais = []
    for _ in range(len(P) // 2):
        pais.append(sample(P, 2))

    return pais

## Algoritmos de recombinação

In [ ]:
from random import randint

def crossover_1_ponto_corte(pais: List[Tuple[Cromossomo, Cromossomo]]):
    filhos = []

    # Crossover - 1 ponto de corte
    for p1, p2 in pais:
        corte = randint(1, len(p1) - 1)

        f1 = p1[:corte] + p2[corte:]
        f2 = p2[:corte] + p1[corte:]

        filhos.append(Cromossomo(p1.fitness, len(f1), f1))
        filhos.append(Cromossomo(p1.fitness, len(f2), f2))

    return filhos

In [ ]:
from random import random

def mutacao_aleatoria(filhos, taxa):

    for cromossomo in filhos:
        for i, pos in enumerate(cromossomo):
            if random() <= taxa:
                cromossomo[i] = 0 if pos else 1

    return filhos

## Algoritmos de seleção dos sobreviventes

In [ ]:
def elitismo(P, F):
    populacao_total = P + F
    populacao_total.sort(key=lambda c: -c.get_fitness())
    return populacao_total[:len(P)]


## Algoritmo genético

In [ ]:
from pprint import pprint

def algoritmo_genetico(tam_populacao,
                       tam_cromossomo,
                       max_geracoes,
                       taxa_mutacao,
                       fitness,
                       selecionar_pais,
                       realizar_crossover,
                       realizar_mutacao,
                       selecionar_sobreviventes):

    # 1. t = 0
    t = 0

    # 2. Inicializar população P0
    P = [Cromossomo(fitness, TAM_CROMOSSOMO) for _ in range(TAM_POPULACAO)]

    # print("# População inicial")
    # pprint([c.vetor for c in P])

    # 3. Enquanto critério de parada == falso
    # TODO: implementar outros critérios de parada
    while t < max_geracoes:

        #   3.1 Avaliar população (Pt)
        #   OK! Avaliação delegada para o cromossomo

        #   3.2 P’ = Selecionar pais (Pt)
        pais = selecionar_pais(P)

        #   3.3 F  = Aplicar recombinação e mutação (P’)
        F = realizar_crossover(pais)
        F = realizar_mutacao(F, TAXA_MUTACAO)

        #   3.4 Avaliar população (F)
        #   OK! Avaliação delegada para o cromossomo

        #   3.5 Pt+1 = Selecionar sobreviventes(Pt + F)
        P = selecionar_sobreviventes(P, F)

        # Imprime o melhor individuo
        print(f'| {t:04d} | {P[0].vetor} | {P[0].get_fitness():4d} |')

        #   3.6 t = t + 1
        t += 1

    print()
    print(f'Melhor solução.: {P[0].vetor}')
    print(f'Fitness........: {P[0].get_fitness()}')

    return P[0]

## Problema 1

Objetivo: maximizar o número de 1s em um cromossomo.

Etapas:

1. Modelar o genótipo e fenótipo do cromossomo
2. Criar a função de avaliação
3. Selecionar parâmetros do AG
4. Executar

In [ ]:
# 1. Modelar o genótipo e fenótipo do cromossomo
# Cromomossomo normal, sem nenhuma codificação especial para esse problema

# 2. Criar a função de avaliação
def fitness_maximizar_1(cromossomo):
    return sum(cromossomo)

# 3. Parametrizar o AG
TAM_POPULACAO = 100
TAM_CROMOSSOMO = 40
MAX_GERACOES = 150

TAXA_MUTACAO = 0.01
TORNEIO = 2

# 4. Executar
solucao = algoritmo_genetico(tam_populacao = TAM_POPULACAO,
                             tam_cromossomo = TAM_CROMOSSOMO,
                             max_geracoes = MAX_GERACOES,
                             taxa_mutacao = TAXA_MUTACAO,
                             fitness = fitness_maximizar_1,
                             selecionar_pais = selecao_aleatoria_com_reposicao,
                             realizar_crossover = crossover_1_ponto_corte,
                             realizar_mutacao = mutacao_aleatoria,
                             selecionar_sobreviventes = elitismo)




| 0000 | [1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1] |   28 |
| 0001 | [1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1] |   28 |
| 0002 | [1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1] |   28 |
| 0003 | [1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1] |   30 |
| 0004 | [1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1] |   30 |
| 0005 | [1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1] |   30 |
| 0006 | [1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1] |   30 |
| 0007 | [1, 1, 1, 1, 0, 1,

In [ ]:
print('== EXPERIMENTO 1 ==')
MAX_GERACOES = 100
TAXA_MUTACAO = 0.9

solucao = algoritmo_genetico(tam_populacao = TAM_POPULACAO,
                             tam_cromossomo = TAM_CROMOSSOMO,
                             max_geracoes = MAX_GERACOES,
                             taxa_mutacao = TAXA_MUTACAO,
                             fitness = fitness_maximizar_1,
                             selecionar_pais = selecao_aleatoria_com_reposicao,
                             realizar_crossover = crossover_1_ponto_corte,
                             realizar_mutacao = mutacao_aleatoria,
                             selecionar_sobreviventes = elitismo)

== EXPERIMENTO 1 ==
| 0000 | [1, 1, 1, 0, 1, 1, 1, 1, 0, 1] |    8 |
| 0001 | [1, 1, 1, 1, 1, 1, 0, 1, 1, 1] |    9 |
| 0002 | [1, 1, 1, 1, 1, 1, 0, 1, 1, 1] |    9 |
| 0003 | [1, 1, 1, 1, 1, 1, 0, 1, 1, 1] |    9 |
| 0004 | [1, 1, 1, 1, 1, 1, 0, 1, 1, 1] |    9 |
| 0005 | [1, 1, 1, 1, 1, 1, 0, 1, 1, 1] |    9 |
| 0006 | [1, 1, 1, 1, 1, 1, 0, 1, 1, 1] |    9 |
| 0007 | [1, 1, 1, 1, 1, 1, 0, 1, 1, 1] |    9 |
| 0008 | [1, 1, 1, 1, 1, 1, 0, 1, 1, 1] |    9 |
| 0009 | [1, 1, 1, 1, 1, 1, 0, 1, 1, 1] |    9 |
| 0010 | [1, 1, 1, 1, 1, 1, 0, 1, 1, 1] |    9 |
| 0011 | [1, 1, 1, 1, 1, 1, 0, 1, 1, 1] |    9 |
| 0012 | [1, 1, 1, 1, 1, 1, 0, 1, 1, 1] |    9 |
| 0013 | [1, 1, 1, 1, 1, 1, 0, 1, 1, 1] |    9 |
| 0014 | [1, 1, 1, 1, 1, 1, 0, 1, 1, 1] |    9 |
| 0015 | [1, 1, 1, 1, 1, 1, 0, 1, 1, 1] |    9 |
| 0016 | [1, 1, 1, 1, 1, 1, 0, 1, 1, 1] |    9 |
| 0017 | [1, 1, 1, 1, 1, 1, 1, 1, 1, 1] |   10 |
| 0018 | [1, 1, 1, 1, 1, 1, 1, 1, 1, 1] |   10 |
| 0019 | [1, 1, 1, 1, 1, 1, 1, 1, 1, 1] |   10 |


In [ ]:
print('== EXPERIMENTO 2 ==')
MAX_GERACOES = 20
TAXA_MUTACAO = 0.1

solucao = algoritmo_genetico(tam_populacao = TAM_POPULACAO,
                             tam_cromossomo = TAM_CROMOSSOMO,
                             max_geracoes = MAX_GERACOES,
                             taxa_mutacao = TAXA_MUTACAO,
                             fitness = fitness_maximizar_1,
                             selecionar_pais = selecao_aleatoria_com_reposicao,
                             realizar_crossover = crossover_1_ponto_corte,
                             realizar_mutacao = mutacao_aleatoria,
                             selecionar_sobreviventes = elitismo)

In [ ]:
print('== EXPERIMENTO 3 ==')
MAX_GERACOES = 2000
TAXA_MUTACAO = 0.1

solucao = algoritmo_genetico(tam_populacao = TAM_POPULACAO,
                             tam_cromossomo = TAM_CROMOSSOMO,
                             max_geracoes = MAX_GERACOES,
                             taxa_mutacao = TAXA_MUTACAO,
                             fitness = fitness_maximizar_1,
                             selecionar_pais = selecao_aleatoria_com_reposicao,
                             realizar_crossover = crossover_1_ponto_corte,
                             realizar_mutacao = mutacao_aleatoria,
                             selecionar_sobreviventes = elitismo)

In [ ]:
from random import sample

def torneio(populacao, tam_torneio):
    populacao = populacao.copy()
    pais = []
    for i in range(len(populacao) // 2):
        tam_torneio = tam_torneio if tam_torneio >= len(populacao) \
                                  else len(populacao)

        individuos = sample(populacao, tam_torneio)
        melhor = individuos[0]
        for individuo in individuos:
            if individuo.get_fitness() > melhor.get_fitness():
                melhor = individuo
        pai1 = melhor
        populacao.remove(pai1)

        individuos = sample(populacao, tam_torneio)
        melhor = individuos[0]
        for individuo in individuos:
            if individuo.get_fitness() > melhor.get_fitness():
                melhor = individuo
        pai2 = melhor

        pais.append((pai1, pai2))
        populacao.remove(pai2)

    return pais

# Atividade

Considere o seguinte problema:

**Problema da fazenda**

Um fazendeiro deseja determinar quantos acres de milho e trigo ele deve plantar esse ano. Um acre de trigo rende 25 sacas e requer 10 horas de trabalho/semana. A saca vale \$4 no mercado. Um acre de milho rende 10 sacas e requer 4 horas de trabalho/semana. A saca vale \$3 no mercado. O governo garante a compra de pelo menos 30 sacas de milho/ano. O fazendeiro dispõe de 7 acres de terra e pode trabalhar 40 horas/semana.

Formule o problema para encontrar a melhor distribuição de acres tal que os ganhos do fazendeiro sejam maximizados. Determine:

* Formulação do problema para ser usado com Algoritmos Genéticos
* Parametrização do algoritmo
* Resultados obtidos
